# Covertype — model training

Reads `training.covertype` (filled by the Airflow DAG), trains 3 candidate models,
saves **all** of them to MinIO as immutable versions, and promotes the best one via
`covertype/production.json` only if it beats the current production model
(champion vs. challenger).

In [1]:
import io
import json
import os
import pickle
import platform
from datetime import datetime, timezone

import boto3
import numpy as np
import pandas as pd
import sklearn
from botocore.exceptions import ClientError
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sqlalchemy import create_engine

# All configuration comes from environment variables set in docker-compose.
DATA_DB_URL = os.environ["DATA_DB_URL"]
MINIO_ENDPOINT = os.environ["MINIO_ENDPOINT"]
MINIO_BUCKET = os.environ["MINIO_BUCKET"]
MODEL_PREFIX = "covertype"
PRODUCTION_POINTER_KEY = f"{MODEL_PREFIX}/production.json"

CATEGORICAL_COLUMNS = ["wilderness_area", "soil_type"]
NUMERIC_COLUMNS = [
    "elevation", "aspect", "slope",
    "horizontal_distance_to_hydrology", "vertical_distance_to_hydrology",
    "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points",
]
FEATURE_COLUMNS = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
TARGET_COLUMN = "cover_type"

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
)

## 1. Load the training-ready data

The train/test split already lives in the table (`split` column, derived from the
row hash), so there is no `train_test_split` here.

In [2]:
engine = create_engine(DATA_DB_URL)
df = pd.read_sql("SELECT * FROM training.covertype", engine)

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
X_train, y_train = train[FEATURE_COLUMNS], train[TARGET_COLUMN]
X_test, y_test = test[FEATURE_COLUMNS], test[TARGET_COLUMN]

print(f"total rows: {len(df)}  |  train: {len(train)}  |  test: {len(test)}")
pd.DataFrame({
    "train": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
}).fillna(0).astype(int)

total rows: 44858  |  train: 35826  |  test: 9032


,train,test
cover_type,,
0,13463,3355
1,15747,4007
2,2902,729
3,20,9
4,636,151
5,2359,596
6,699,185


In [3]:
# Sanity check: training on data that is missing a class would produce a model
# that can never predict it. The data comes from a sampled API, so check it.
missing = sorted(set(range(7)) - set(y_train.unique()))
assert not missing, f"Classes missing from training data: {missing} — collect more data first."
print("All 7 classes present in the training split.")

All 7 classes present in the training split.


## 2. Candidate pipelines

Each candidate is a full `Pipeline`: preprocessing + model in a single object, so the
saved model accepts raw values like `"Rawah"` / `"C7202"` directly.

- `OneHotEncoder(handle_unknown="ignore")`: an unseen soil type at inference time
  produces all-zero columns instead of an error.
- `sparse_output=False`: `HistGradientBoostingClassifier` does not accept sparse input.
- `class_weight="balanced"`: rare classes (class 3 has only a few dozen rows) weigh
  more in training.

In [4]:
def make_preprocessor():
    return ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLUMNS),
        ("numeric", StandardScaler(), NUMERIC_COLUMNS),
    ])


candidates = {
    "random_forest": RandomForestClassifier(
        n_estimators=100, min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=42
    ),
    "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "hist_gradient_boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

pipelines = {name: Pipeline([("preprocess", make_preprocessor()), ("model", model)])
             for name, model in candidates.items()}
list(pipelines)

['random_forest', 'logistic_regression', 'hist_gradient_boosting']

## 3. Train and evaluate

Model selection uses **macro-F1** (every class counts equally), not accuracy.

In [5]:
def evaluate(pipeline, X, y):
    y_pred = pipeline.predict(X)
    return {
        "macro_f1": round(float(f1_score(y, y_pred, average="macro")), 4),
        "accuracy": round(float(accuracy_score(y, y_pred)), 4),
        "per_class": classification_report(y, y_pred, output_dict=True, zero_division=0),
    }


results = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    results[name] = evaluate(pipeline, X_test, y_test)
    print(f"{name:24s} macro-F1={results[name]['macro_f1']:.4f}  accuracy={results[name]['accuracy']:.4f}")

best_name = max(results, key=lambda n: results[n]["macro_f1"])
print(f"\nbest candidate: {best_name}")

random_forest            macro-F1=0.8504  accuracy=0.9391


logistic_regression      macro-F1=0.5309  accuracy=0.6638


hist_gradient_boosting   macro-F1=0.8178  accuracy=0.9059

best candidate: random_forest


## 4. Save every candidate to MinIO

Each candidate becomes its own immutable version: `covertype/<timestamp>_<model>/`.
`metrics.json` records the library versions — the inference API refuses to start if
its scikit-learn version differs.

In [6]:
run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
library_versions = {
    "python": platform.python_version(),
    "scikit-learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}


def put_json(key, payload):
    s3.put_object(Bucket=MINIO_BUCKET, Key=key, Body=json.dumps(payload, indent=2).encode(),
                  ContentType="application/json")


saved_versions = {}
for name, pipeline in pipelines.items():
    version = f"{run_timestamp}_{name}"
    model_key = f"{MODEL_PREFIX}/{version}/model.pkl"
    metrics_key = f"{MODEL_PREFIX}/{version}/metrics.json"

    s3.put_object(Bucket=MINIO_BUCKET, Key=model_key, Body=pickle.dumps(pipeline))
    put_json(metrics_key, {
        "version": version,
        "model_name": name,
        "trained_at": run_timestamp,
        "metrics": results[name],
        "rows": {"train": len(train), "test": len(test)},
        "class_distribution_train": {int(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
        "feature_columns": FEATURE_COLUMNS,
        "library_versions": library_versions,
    })
    saved_versions[name] = {"version": version, "model_key": model_key, "metrics_key": metrics_key}
    print(f"saved {model_key}")

saved covertype/20260927T000248Z_random_forest/model.pkl
saved covertype/20260927T000248Z_logistic_regression/model.pkl
saved covertype/20260927T000248Z_hist_gradient_boosting/model.pkl


## 5. Champion vs. challenger

The current production model is re-scored on **today's** test split next to the new
best candidate — the test split grows with every collection, so comparing against the
number stored in an old `metrics.json` would compare different test sets.
`production.json` is written **last**, after the model it points to is fully uploaded.

In [7]:
def load_production_pointer():
    try:
        body = s3.get_object(Bucket=MINIO_BUCKET, Key=PRODUCTION_POINTER_KEY)["Body"].read()
        return json.loads(body)
    except ClientError as error:
        if error.response["Error"]["Code"] == "NoSuchKey":
            return None
        raise


challenger_score = results[best_name]["macro_f1"]
pointer = load_production_pointer()

if pointer is None:
    promote = True
    print(f"No production model yet -> promoting {best_name} (macro-F1 {challenger_score:.4f}).")
else:
    champion_bytes = s3.get_object(Bucket=MINIO_BUCKET, Key=pointer["model_key"])["Body"].read()
    champion = pickle.loads(champion_bytes)
    champion_score = evaluate(champion, X_test, y_test)["macro_f1"]
    promote = challenger_score > champion_score
    print(f"champion   {pointer['version']}: macro-F1 {champion_score:.4f} (on today's test split)")
    print(f"challenger {saved_versions[best_name]['version']}: macro-F1 {challenger_score:.4f}")
    print("-> promoting challenger" if promote else "-> keeping champion")

if promote:
    put_json(PRODUCTION_POINTER_KEY, {
        **saved_versions[best_name],
        "model_name": best_name,
        "macro_f1_at_promotion": challenger_score,
        "promoted_at": datetime.now(timezone.utc).isoformat(),
        "library_versions": library_versions,
    })
    print(f"production.json -> {saved_versions[best_name]['version']}")

champion   20260927T000102Z_random_forest: macro-F1 0.8504 (on today's test split)
challenger 20260927T000248Z_random_forest: macro-F1 0.8504
-> keeping champion


In [8]:
# What's in the bucket now.
listing = s3.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=f"{MODEL_PREFIX}/")
for obj in listing.get("Contents", []):
    print(f"{obj['Size'] / 1e6:8.2f} MB  {obj['Key']}")

    0.00 MB  covertype/20260927T000102Z_hist_gradient_boosting/metrics.json
    0.79 MB  covertype/20260927T000102Z_hist_gradient_boosting/model.pkl
    0.00 MB  covertype/20260927T000102Z_logistic_regression/metrics.json
    0.01 MB  covertype/20260927T000102Z_logistic_regression/model.pkl
    0.00 MB  covertype/20260927T000102Z_random_forest/metrics.json
   41.59 MB  covertype/20260927T000102Z_random_forest/model.pkl
    0.00 MB  covertype/20260927T000248Z_hist_gradient_boosting/metrics.json
    0.79 MB  covertype/20260927T000248Z_hist_gradient_boosting/model.pkl
    0.00 MB  covertype/20260927T000248Z_logistic_regression/metrics.json
    0.01 MB  covertype/20260927T000248Z_logistic_regression/model.pkl
    0.00 MB  covertype/20260927T000248Z_random_forest/metrics.json
   41.59 MB  covertype/20260927T000248Z_random_forest/model.pkl
    0.00 MB  covertype/production.json
